In [47]:
import os
import re
import sys
import time
from pathlib import Path
from typing import List, Tuple
import json

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.action_chains import ActionChains
from urllib.parse import urljoin
import random
from bs4 import BeautifulSoup

In [48]:
# class kami_crawler:
#     def __init__(
#             self,
#             url: str,
#             out_dir: str,
#             headless: bool=True,
#             wait_s: int=12,
#     ) -> None:
#         self.url = url
#         self.out_dir = out_dir
#         self.headless = headless
#         self.wait_s = wait_s

#         opts = webdriver.ChromeOptions()
        
#         chrome_options = os.getenv('CHROME_OPTIONS', '')
#         if chrome_options:
#             for option in chrome_options.split():
#                 opts.add_argument(option)
#         else:
#             # Default options
#             if headless:
#                 opts.add_argument("--headless=new")
#             opts.add_argument("--no-sandbox")
#             opts.add_argument("--disable-dev-shm-usage")
#             opts.add_argument("--disable-gpu")
#             opts.add_argument("--window-size=1280,1800")
#             opts.add_argument("--disable-extensions")
#             opts.add_argument("--disable-plugins")
#             opts.add_argument("--disable-images")  # Speed up crawling

#         self.driver = webdriver.Chrome(
#             service=Service(ChromeDriverManager().install()), options=opts
#         )
#         self.wait = WebDriverWait(self.driver, wait_s)

#     def _ready(self):
#         self.wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
    
#     def _visible(self, by, sel):
#         return self.wait.until(EC.visibility_of_element_located((by, sel)))

#     def extract_kami_list(self):
#         self.driver.get(self.url)
#         self._ready()

#         kami = []

#         rows = self.driver.find_elements(
#             By.CSS_SELECTOR, "#sortabletable1 tbody tr"
#         )
#         print('Đang lấy danh sách các kami...')

#         for row in rows:
#             try:
#                 link_elements = row.find_elements(By.CSS_SELECTOR, "td:first-child a")

#                 for link_element in link_elements:
#                     name = link_element.text.strip()
#                     link = link_element.get_attribute("href")

#                     #link = urljoin(BASE_URL, link)

#                     kami.append({
#                         "name": name,
#                         "link": link
#                     })

#             except:
#                 continue
#         print('Đã lấy toàn bộ danh sách của ' + str(len(kami)) + ' kami!')

#         return kami

#     def parse_normal_kami(self, table):
#         rows = table.find_elements(By.CSS_SELECTOR, "tr")

#         info = {}
#         skills = []
#         flavor = ""

#         mode = "info"  # info | skill | flavor
#         headers = []
#         current_type = None

#         for row in rows:
#             text = row.text.strip()

#             # 🔹 chuyển mode
#             if "バースト/アビリティ" in text:
#                 mode = "skill"
#                 continue

#             if "フレーバーテキスト" in text:
#                 mode = "flavor"
#                 continue

#             ths = row.find_elements(By.TAG_NAME, "th")
#             tds = row.find_elements(By.TAG_NAME, "td")

#             # =========================
#             # 🟦 INFO BLOCK
#             # =========================
#             if mode == "info":

#                 i = 0
#                 while i < len(rows):
#                     row = rows[i]

#                     ths = row.find_elements(By.TAG_NAME, "th")
#                     tds = row.find_elements(By.TAG_NAME, "td")

#                     th_texts = [th.text.strip() for th in ths if th.text.strip()]
#                     td_texts = [td.text.strip() for td in tds if td.text.strip()]

#                     # =========================
#                     # 🟦 TITLE
#                     # =========================
#                     if not info.get("name") and th_texts:
#                         # if "］" in th_texts[0]:
#                         #     info["name"] = th_texts[0]
#                         #     i += 1
#                         #     continue
#                         for th in ths:
#                             text = th.text.strip()

#                             # lấy name
#                             if text and "］" in text:
#                                 info["name"] = text

#                             # lấy image
#                             imgs = th.find_elements(By.TAG_NAME, "img")
#                             if imgs:
#                                 info["image"] = imgs[0].get_attribute("src")

#                         i += 1
#                         continue

#                     # =========================
#                     # 🟨 KEY row → look ahead VALUE row
#                     # =========================
#                     if th_texts and not td_texts:

#                         # tránh nhảy vào skill table
#                         if "種類" in th_texts or "バースト" in th_texts:
#                             break

#                         # look next row
#                         if i + 1 < len(rows):
#                             next_row = rows[i + 1]
#                             next_tds = next_row.find_elements(By.TAG_NAME, "td")
#                             next_td_texts = [td.text.strip() for td in next_tds if td.text.strip()]

#                             if next_td_texts:
#                                 # 1 key
#                                 if len(th_texts) == 1:
#                                     info[th_texts[0]] = next_td_texts[0]

#                                 # 2 key (MinHP / MaxHP)
#                                 else:
#                                     for k, v in zip(th_texts, next_td_texts):
#                                         info[k] = v

#                                 i += 2
#                                 continue

#                     i += 1

#             # =========================
#             # 🟨 SKILL BLOCK
#             # =========================
#             elif mode == "skill":

#                 # header
#                 if len(ths) == 6:
#                     headers = [th.text.strip() for th in ths]
#                     continue

#                 if not headers:
#                     continue

#                 row_data = []

#                 # xử lý loại (rowspan)
#                 if len(ths) >= 1:
#                     current_type = ths[0].text.strip()
#                     row_data.append(current_type)
#                 else:
#                     row_data.append(current_type)

#                 # xử lý td + colspan
#                 for td in tds:
#                     value = td.text.strip()
#                     colspan = td.get_attribute("colspan")

#                     if colspan:
#                         row_data.extend([value] * int(colspan))
#                     else:
#                         row_data.append(value)

#                 row_data = row_data[:len(headers)]

#                 skills.append(dict(zip(headers, row_data)))

#             # =========================
#             # 🟥 FLAVOR
#             # =========================
#             elif mode == "flavor":
#                 if tds:
#                     flavor = tds[0].text.strip()

#         return {
#             "info": info,
#             "skills": skills,
#             "flavor": flavor
#         }

#     def parse_normal_kami_v2(self, table):
#         # 1. Chuyển đổi toàn bộ thẻ tr, td, th trong bảng thành Ma trận Text
#         rows = table.find_elements(By.TAG_NAME, "tr")
#         matrix = []
#         for row in rows:
#             cells = row.find_elements(By.XPATH, "./td | ./th")
#             matrix.append([cell.text.strip() for cell in cells])

#         # 2. Lấy link ảnh (nằm ở thẻ img duy nhất trong bảng này)
#         images = [img.get_attribute("src") for img in table.find_elements(By.TAG_NAME, "img")]
#         image_url = images[0] if images else ""

#         # --- Các hàm Helper để định vị dòng dựa trên Keyword ---
#         def find_row_idx(keyword):
#             for idx, r in enumerate(matrix):
#                 if any(keyword in cell for cell in r):
#                     return idx
#             return -1

#         def get_value_below(keyword):
#             idx = find_row_idx(keyword)
#             if idx != -1 and idx + 1 < len(matrix):
#                 # Lấy phần tử đầu tiên của dòng ngay phía dưới tiêu đề
#                 return matrix[idx + 1][0]
#             return ""

#         # 3. Trích xuất phần "info" dựa trên các Keyword nhãn cố định
#         # Đối với các nhãn đi đôi song song (MinHP/MaxHP), ta lấy trực tiếp chỉ số cột dòng dưới
#         hp_idx = find_row_idx("MinHP")
#         min_hp = matrix[hp_idx + 1][0] if hp_idx != -1 else ""
#         max_hp = matrix[hp_idx + 1][1] if hp_idx != -1 and len(matrix[hp_idx + 1]) > 1 else ""

#         atk_idx = find_row_idx("Min攻撃力")
#         min_atk = matrix[atk_idx + 1][0] if atk_idx != -1 else ""
#         max_atk = matrix[atk_idx + 1][1] if atk_idx != -1 and len(matrix[atk_idx + 1]) > 1 else ""

#         wpn_idx = find_row_idx("解放ウェポン")
#         release_wpn = matrix[wpn_idx + 1][0] if wpn_idx != -1 else ""
#         expert_wpn = matrix[wpn_idx + 1][1] if wpn_idx != -1 and len(matrix[wpn_idx + 1]) > 1 else ""

#         info = {
#             "name": matrix[0][0],  # Tên luôn nằm ở ô đầu tiên của dòng đầu tiên
#             "image": image_url,
#             "レアリティ": get_value_below("レアリティ"),
#             "属性": get_value_below("属性"),
#             "MaxLv": get_value_below("MaxLv"),
#             "タイプ": get_value_below("タイプ"),
#             "MinHP": min_hp,
#             "MaxHP\n覚醒後": max_hp,  # Key đồng bộ theo dạng \n của bạn
#             "Min攻撃力": min_atk,
#             "Max攻撃力\n覚醒後": max_atk,
#             "解放ウェポン": release_wpn,
#             "得意ウェポン": expert_wpn
#         }

#         # 4. Trích xuất danh sách "skills"
#         skills = []
#         # Điểm bắt đầu là dòng ngay sau dòng tiêu đề cột của kỹ năng (種類, 名称, 習得...)
#         start_skills_idx = find_row_idx("種類") + 1
#         end_skills_idx = find_row_idx("フレーバーテキスト")

#         current_type = "" # Biến dùng để ghi nhớ Loại (种类) khi gặp rowspan
        
#         if start_skills_idx > 0 and end_skills_idx != -1:
#             for i in range(start_skills_idx, end_skills_idx):
#                 row = matrix[i]
#                 if not row:
#                     continue

#                 # Trường hợp 1: Dòng có chứa từ khóa Loại kỹ năng (Mở đầu cụm rowspan)
#                 if any(kw in row[0] for kw in ["バースト", "アビリティ", "アシst", "アシスト"]):
#                     current_type = row[0]
                    
#                     # Kiểm tra độ dài xem dòng này có chứa data kỹ năng không (bỏ qua dòng tiêu đề phụ nếu có)
#                     if len(row) >= 3:
#                         # Nếu dòng có 6 cột đầy đủ [種類, 名称, 習得, 使用, 効果, スキル効果]
#                         if len(row) == 6:
#                             skill_obj = {
#                                 "種類": current_type,
#                                 "名称": row[1],
#                                 "習得/+化Lv": row[2],
#                                 "使用間隔": row[3],
#                                 "効果時間": row[4],
#                                 "スキル効果": row[5]
#                             }
#                         else:
#                             # Bảng biến thể gộp cột (ví dụ dòng Burst đầu tiên gộp 3 ô Lv/Sử dụng/Thời gian thành "-")
#                             skill_obj = {
#                                 "種類": current_type,
#                                 "名称": row[1],
#                                 "習得/+化Lv": row[2] if len(row) > 3 else "-",
#                                 "使用間隔": row[2] if len(row) > 3 else "-",
#                                 "効果時間": row[2] if len(row) > 3 else "-",
#                                 "スキル効果": row[-1]
#                             }
#                         skills.append(skill_obj)

#                 # Trường hợp 2: Dòng dữ liệu phụ ăn theo rowspan (Không có cột Loại ở đầu)
#                 elif len(row) >= 2:
#                     # Cấu trúc chuẩn của dòng phụ: [名称, 習得, 使用, 効果, スキル効果] -> 5 cột
#                     if len(row) == 5:
#                         skill_obj = {
#                             "種類": current_type,
#                             "名称": row[0],
#                             "習得/+化Lv": row[1],
#                             "使用間隔": row[2],
#                             "効果時間": row[3],
#                             "スキル効果": row[4]
#                         }
#                     else:
#                         # Gộp ô thiếu cột (ví dụ dòng Burst nâng cấp giải phóng bằng đột phá giới hạn)
#                         skill_obj = {
#                             "種類": current_type,
#                             "名称": row[0],
#                             "習得/+化Lv": row[1] if len(row) > 2 else "-",
#                             "使用間隔": row[1] if len(row) > 2 else "-",
#                             "効果時間": row[1] if len(row) > 2 else "-",
#                             "スキル効果": row[-1]
#                         }
#                     skills.append(skill_obj)

#         # 5. Lấy thông tin "flavor" (Nằm ngay dưới hàng chữ フレーバーテキスト)
#         flavor_idx = find_row_idx("フレーバーテキスト")
#         flavor_text = matrix[flavor_idx + 1][0] if flavor_idx != -1 and flavor_idx + 1 < len(matrix) else ""

#         # Trả về kết quả hoàn chỉnh trùng khớp định dạng mẫu
#         return {
#             "info": info,
#             "skills": skills,
#             "flavor": flavor_text
#         }
    
#     def parse_awaken_kami(self, table):
#         rows = table.find_elements(By.TAG_NAME, "tr")
        
#         # Chuyển đổi toàn bộ bảng thành một list các list (text) để dễ xử lý logic
#         matrix = []
#         for row in rows:
#             cells = row.find_elements(By.XPATH, "./td | ./th")
#             matrix.append([cell.text.strip() for cell in cells])

#         def find_row_by_text(search_text, start_index=0):
#             for i in range(start_index, len(matrix)):
#                 if search_text in matrix[i]:
#                     return i
#             return -1

#         # 1. Lấy thông tin cơ bản (Dòng đầu luôn là tên, dòng 2 là ảnh)
#         names = matrix[0]
#         # Lấy link ảnh trực tiếp từ driver vì matrix chỉ lưu text
#         images = [img.get_attribute("src") for img in rows[1].find_elements(By.TAG_NAME, "img")]

#         # 2. Tìm dòng chứa chỉ số (Rarity/HP)
#         idx_stats = find_row_by_text("レアリティ")
#         stats_val = matrix[idx_stats + 1]
#         stats_val2 = matrix[idx_stats + 3]

#         # 3. Hàm xử lý kỹ năng (Burst/Ability/Assist)
#         def extract_skills_sector(start_keyword, end_keyword, current_search_idx):
#             skills = []
#             start_idx = find_row_by_text(start_keyword, current_search_idx)
#             end_idx = find_row_by_text(end_keyword, start_idx)
            
#             if start_idx != -1:
#                 # Lấy các dòng nằm giữa tiêu đề Skill và tiêu đề tiếp theo
#                 # Bỏ qua dòng header của chính nó (start_idx + 1)
#                 for i in range(start_idx + 2, end_idx if end_idx != -1 else len(matrix)):
#                     # Nếu dòng này chứa tiêu đề lớn khác thì dừng lại
#                     if any(x in matrix[i] for x in ["バースト", "アビリティ", "アシスト", "フレーバーテキスト"]):
#                         break
#                     skills.append(matrix[i])
#             return skills, start_idx

#         # --- Tách dữ liệu cho BASE và AWAKEN ---
#         # Tìm mốc phân chia giữa Base và Awaken
#         print(stats_val)
#         print(stats_val2)
#         split_idx = find_row_by_text("バースト/アビリティ [覚醒後]")

#         # BASE DATA
#         hp_split = stats_val[-1].split(" / ")
#         atk_split = stats_val2[-1].split(" / ")
        
#         base_skills = []
#         # Gom tất cả kỹ năng trước mốc split_idx
#         for kw in ["バースト", "アビリティ", "アシスト"]:
#             s, _ = extract_skills_sector(kw, "フレーバーテキスト", 0)
#             base_skills.extend(s)

#         base_data = {
#             'name': names[0],
#             'image': images[0],
#             'レアリティ': stats_val[0],
#             'MaxHP/覚醒後': hp_split[0],
#             'Max攻撃力': atk_split[0],
#             'skills': base_skills,
#             'flavor': matrix[find_row_by_text("フレーバーテキスト", 0) + 1][0]
#         }

#         # AWAKEN DATA
#         awaken_data = None
#         if split_idx != -1:
#             awaken_skills = []
#             for kw in ["バースト", "アビリティ", "アシスト"]:
#                 s, _ = extract_skills_sector(kw, "フレーバーテキスト", split_idx)
#                 awaken_skills.extend(s)
                
#             awaken_data = {
#                 'name': names[1] if len(names) > 1 else names[0] + "[覚醒]",
#                 'image': images[1] if len(images) > 1 else images[0],
#                 'MaxHP/覚醒後': hp_split[1] if len(hp_split) > 1 else hp_split[0],
#                 'Max攻撃力': atk_split[1] if len(atk_split) > 1 else atk_split[0],
#                 'skills': awaken_skills,
#                 'flavor': matrix[find_row_by_text("フレーバーテキスト", split_idx) + 1][0]
#             }

#         return {'base': base_data, 'awaken': awaken_data}

#     def parse_awaken_kami_v2(self, table):
#         # Chuyển bảng thành matrix để truy xuất nội dung
#         def find_row_by_text(matrix, search_text, start_index=0):
#             for i in range(start_index, len(matrix)):
#                 if search_text in matrix[i]:
#                     return i
#             return -1
#         rows = table.find_elements(By.TAG_NAME, "tr")
#         matrix = []
#         for row in rows:
#             cells = row.find_elements(By.XPATH, "./td | ./th")
#             matrix.append([cell.text.strip() for cell in cells])

#         # Lấy danh sách ảnh (Gaia có 2 ảnh: Base và Awaken)
#         images = [img.get_attribute("src") for img in table.find_elements(By.TAG_NAME, "img")]
        
#         # Tìm các mốc quan trọng
#         idx_stats = find_row_by_text(matrix, "レアリティ")
#         idx_flavor_1 = find_row_by_text(matrix, "フレーバーテキスト")
#         idx_split = find_row_by_text(matrix, "[覚醒後]")
#         idx_flavor_2 = find_row_by_text(matrix, "フレーバーテキスト", idx_split)

#         def split_val(text, index):
#             parts = text.split('/')
#             if len(parts) > index:
#                 return parts[index].strip()
#             return parts[0].strip()

#         def get_skills(start_idx, end_idx):
#             skills = []
#             current_type = ""
#             for i in range(start_idx, end_idx):
#                 row = matrix[i]
#                 # Nếu dòng có 'バースト', 'アビリティ', 'アシスト' thì đó là cột loại
#                 if any(x in row[0] for x in ["バースト", "アビリティ", "アシスト"]):
#                     current_type = row[0]
#                     # Format lại skills theo đúng các key bạn yêu cầu
#                     # Lưu ý: bảng Awaken thường gộp cột, nên index có thể khác bảng Normal
#                     if len(row) >= 3: # Dòng chứa data
#                         # Bảng Awaken: [Loại, Tên, (Lv), (Time), Effect]
#                         # Tùy vào colspan mà row[1], row[2] sẽ chứa data khác nhau
#                         skill_obj = {
#                             "種類": current_type,
#                             "名称": row[1] if len(row) > 1 else "",
#                             "習得/+化Lv": row[2] if len(row) > 5 else "-",
#                             "使用間隔": row[3] if len(row) > 5 else "-",
#                             "効果時間": row[4] if len(row) > 5 else "-",
#                             "スキル効果": row[-1]
#                         }
#                         skills.append(skill_obj)
#                 elif len(row) > 1 and current_type != "": # Các dòng phụ của cùng 1 loại
#                     skill_obj = {
#                         "種類": current_type,
#                         "名称": row[0],
#                         "習得/+化Lv": row[1] if len(row) > 4 else "-",
#                         "使用間隔": row[2] if len(row) > 4 else "-",
#                         "効果時間": row[3] if len(row) > 4 else "-",
#                         "スキル効果": row[-1]
#                     }
#                     skills.append(skill_obj)
#             return skills

#         # --- TẠO DATA CHO BASE ---
#         base_data = {
#             "info": {
#                 "name": matrix[0][0],
#                 "image": images[0] if len(images) > 0 else "",
#                 "レアリティ": matrix[idx_stats + 1][0],
#                 "属性": matrix[idx_stats + 1][1],
#                 "MaxLv": matrix[idx_stats + 1][2],
#                 "タイプ": matrix[idx_stats + 2][0],
#                 "MinHP": matrix[idx_stats + 1][3],
#                 "MaxHP\n覚醒後": split_val(matrix[idx_stats + 1][4], 0),
#                 "Min攻撃力": matrix[idx_stats + 2][3],
#                 "Max攻撃力\n覚醒後": split_val(matrix[idx_stats + 2][4], 0),
#                 "解放ウェポン": matrix[idx_stats + 2][1],
#                 "得意ウェポン": split_val(matrix[idx_stats + 2][2], 0)
#             },
#             "skills": get_skills(idx_stats + 3, idx_flavor_1),
#             "flavor": matrix[idx_flavor_1 + 1][0] if idx_flavor_1 != -1 else ""
#         }

#         # --- TẠO DATA CHO AWAKEN ---
#         awaken_data = {
#             "info": {
#                 "name": matrix[0][1] if len(matrix[0]) > 1 else matrix[0][0] + "[神化覚醒]",
#                 "image": images[1] if len(images) > 1 else images[0],
#                 "レアリティ": matrix[idx_stats + 1][0],
#                 "属性": matrix[idx_stats + 1][1],
#                 "MaxLv": matrix[idx_stats + 1][2],
#                 "タイプ": matrix[idx_stats + 2][0],
#                 "MinHP": matrix[idx_stats + 1][3],
#                 "MaxHP\n覚醒後": split_val(matrix[idx_stats + 1][4], 1),
#                 "Min攻撃力": matrix[idx_stats + 2][3],
#                 "Max攻撃力\n覚醒後": split_val(matrix[idx_stats + 2][4], 1),
#                 "解放ウェポン": matrix[idx_stats + 2][1],
#                 "得意ウェポン": split_val(matrix[idx_stats + 2][2], 1)
#             },
#             "skills": get_skills(idx_split + 1, idx_flavor_2),
#             "flavor": matrix[idx_flavor_2 + 1][0] if idx_flavor_2 != -1 else ""
#         }

#         return {'base': base_data, 'awaken': awaken_data}
    
#     def extract_all_kami_data(self):
#         kami_list = self.extract_kami_list()
#         all_data = []
#         os.makedirs(self.out_dir, exist_ok=True)
#         path = os.path.join(self.out_dir, 'all_kami_data.jsonl')
        
#         for i in range(len(kami_list)):
#             time.sleep(random.uniform(0.8,1.6))
#             self.driver.get(kami_list[i]['link'])
#             self._ready()
#             try:
#                 table = self.driver.find_element(By.CSS_SELECTOR, "table.style_table")
#                 has_awaken_tag = len(table.find_elements(By.CSS_SELECTOR, "th span, th")) > 0 and \
#                     any("覚醒" in el.text for el in table.find_elements(By.CSS_SELECTOR, "th"))
#                 print(has_awaken_tag)
                
#                 print(f'Đang lấy dữ liệu của kami thứ {i + 1}')

#                 if has_awaken_tag:
#                     kami_data = self.parse_awaken_kami_v2(table)
#                 else:
#                     kami_data = self.parse_normal_kami_v2(table)
#                 #all_data.append(kami_data)
#                 with open(path, "a", encoding="utf-8") as f:
#                     #json.dump(kami_data, f, ensure_ascii=False, indent=2)
#                     f.write(json.dumps(kami_data, ensure_ascii=False) + "\n")
#             except ValueError as e:
#                 print(f'Gặp lỗi {e} khi lấy data của kami thứ {i}')

In [49]:
class kami_crawler:
    def __init__(
            self,
            url: str,
            out_dir: str,
            headless: bool=True,
            wait_s: int=12,
    ) -> None:
        self.url = url
        self.out_dir = out_dir
        self.headless = headless
        self.wait_s = wait_s

        opts = webdriver.ChromeOptions()
        
        chrome_options = os.getenv('CHROME_OPTIONS', '')
        if chrome_options:
            for option in chrome_options.split():
                opts.add_argument(option)
        else:
            # Default options
            if headless:
                opts.add_argument("--headless=new")
            opts.add_argument("--no-sandbox")
            opts.add_argument("--disable-dev-shm-usage")
            opts.add_argument("--disable-gpu")
            opts.add_argument("--window-size=1280,1800")
            opts.add_argument("--disable-extensions")
            opts.add_argument("--disable-plugins")
            opts.add_argument("--disable-images")  # Speed up crawling

        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=opts
        )
        self.wait = WebDriverWait(self.driver, wait_s)

    def _ready(self):
        self.wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
    
    def _visible(self, by, sel):
        return self.wait.until(EC.visibility_of_element_located((by, sel)))

    def extract_kami_list(self):
        self.driver.get(self.url)
        self._ready()

        kami = []

        rows = WebDriverWait(self.driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "tbody[aria-live='polite'] tr"))
        )
        print('Đang lấy danh sách các kami...')

        for row in rows:
            try:
                link_elements = row.find_elements(By.CSS_SELECTOR, "td:nth-of-type(2) a.rel-wiki-page")

                for link_element in link_elements:
                    name = link_element.text.strip()
                    link = link_element.get_attribute("href")

                    #link = urljoin(BASE_URL, link)

                    kami.append({
                        "name": name,
                        "link": link
                    })

            except:
                continue
        print('Đã lấy toàn bộ danh sách của ' + str(len(kami)) + ' kami!')

        return kami

    def parse_kami_data(self, html_content, kami_name):
        soup = BeautifulSoup(html_content, 'html.parser')

        result = {
            'info': {'name': kami_name},
            'skill': [],
            'flavor': ''
        }

        # 1. Duyệt qua tất cả các thẻ <tr> trong bảng
        rows = soup.find_all('tr')
        current_skill_type = None

        for row in rows:
            ths = row.find_all('th', recursive=False)
            tds = row.find_all('td', recursive=False)

            # BẮT TRƯỜNG HỢP FLAVOR TEXT: Dòng chỉ có 1 thẻ <td> và colspan="6"
            if len(tds) == 1 and tds[0].get('colspan') == '6':
                # Dùng separator='\n' để thay thế thẻ <br> thành dấu xuống dòng hợp lệ
                result['flavor'] = tds[0].get_text(separator='\n', strip=True)
                continue

            # XỬ LÝ CÁC DÒNG CÓ TIÊU ĐỀ (TH)
            if ths:
                header_text = ths[0].get_text(strip=True)
                
                # Lấy ảnh nhân vật từ dòng "基本情報"
                if header_text == '基本情報' and len(tds) > 0:
                    img_tag = tds[0].find('img')
                    if img_tag:
                        result['info']['img'] = img_tag['src']
                    continue
                    
                # Xác định các headers của skill để làm mốc
                if 'バースト' in header_text:
                    current_skill_type = 'バースト'
                    continue
                elif 'アビリティ' in header_text:
                    current_skill_type = 'アビリティ'
                    continue
                elif 'アシスト' in header_text:
                    current_skill_type = 'アシスト'
                    continue
                    
                # Cập nhật thông tin cơ bản (Info) - Trừ các dòng header kỹ năng
                if len(ths) == 1 and len(tds) == 1:
                    key = header_text
                    val = tds[0].get_text(strip=True)
                    
                    # Ép kiểu int cho level theo như format mẫu của bạn
                    if key == '最大レベル':
                        val = int(val)
                        
                    result['info'][key] = val
                    continue

            # XỬ LÝ CÁC DÒNG CHỨA DỮ LIỆU SKILL (Chỉ có TD, không có TH)
            if current_skill_type and not ths and len(tds) >= 4:
                skill_dict = {}
                
                # tds[0] là ảnh icon (bỏ qua), tds[1] là tên skill
                skill_dict[current_skill_type] = tds[1].get_text(strip=True)
                
                # tds[2] là điều kiện, nếu trống gán bằng '-'
                condition = tds[2].get_text(strip=True)
                skill_dict['習得条件'] = condition if condition else '-'

                # Phân loại theo skill type vì cột của Ability dài hơn
                if current_skill_type == 'アビリティ' and len(tds) >= 6:
                    skill_dict['使用間隔'] = tds[3].get_text(strip=True)
                    skill_dict['効果時間'] = tds[4].get_text(strip=True)
                    skill_dict['効果'] = tds[5].get_text(strip=True)
                else:
                    # Đối với Burst và Assist
                    skill_dict['効果'] = tds[3].get_text(strip=True)
                    
                result['skill'].append(skill_dict)
            
        return result

    def extract_kami_data(self):
        kami_list = self.extract_kami_list()
        all_data = []
        os.makedirs(self.out_dir, exist_ok=True)
        path = os.path.join(self.out_dir, 'all_kami_data.jsonl')
        
        for i in range(len(kami_list)):
            time.sleep(random.uniform(0.8,1.6))
            self.driver.get(kami_list[i]['link'])
            self._ready()
            kami_name = kami_list[i]['name']
            try:
                print(f'Đang lấy dữ liệu của kami thứ {i + 1}')
                tables = WebDriverWait(self.driver, 10).until(
                    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.h-scrollable")))
                table_html = tables[0].get_attribute("innerHTML")
                kami_data = self.parse_kami_data(table_html, kami_name)
                with open(path, "a", encoding="utf-8") as f:
                    f.write(json.dumps(kami_data, ensure_ascii=False) + "\n")
            except ValueError as e:
                print(f'Gặp lỗi {e} khi lấy data của kami thứ {i}')

In [50]:
if __name__ == "__main__":
    from dotenv import load_dotenv
    load_dotenv()
    test = kami_crawler(r'https://wikiwiki.jp/kamiprodb/%E7%A5%9E%E5%A7%AB/SSR/%E7%81%AB', "kami", False, 10)
    test.extract_kami_data()

Đang lấy danh sách các kami...
Đã lấy toàn bộ danh sách của 106 kami!
Đang lấy dữ liệu của kami thứ 1
Đang lấy dữ liệu của kami thứ 2
Đang lấy dữ liệu của kami thứ 3
Đang lấy dữ liệu của kami thứ 4
Đang lấy dữ liệu của kami thứ 5
Đang lấy dữ liệu của kami thứ 6
Đang lấy dữ liệu của kami thứ 7
Đang lấy dữ liệu của kami thứ 8
Đang lấy dữ liệu của kami thứ 9
Đang lấy dữ liệu của kami thứ 10
Đang lấy dữ liệu của kami thứ 11
Đang lấy dữ liệu của kami thứ 12
Đang lấy dữ liệu của kami thứ 13
Đang lấy dữ liệu của kami thứ 14
Đang lấy dữ liệu của kami thứ 15
Đang lấy dữ liệu của kami thứ 16
Đang lấy dữ liệu của kami thứ 17
Đang lấy dữ liệu của kami thứ 18
Đang lấy dữ liệu của kami thứ 19
Đang lấy dữ liệu của kami thứ 20
Đang lấy dữ liệu của kami thứ 21
Đang lấy dữ liệu của kami thứ 22
Đang lấy dữ liệu của kami thứ 23
Đang lấy dữ liệu của kami thứ 24
Đang lấy dữ liệu của kami thứ 25
Đang lấy dữ liệu của kami thứ 26
Đang lấy dữ liệu của kami thứ 27
Đang lấy dữ liệu của kami thứ 28
Đang lấy dữ liệ

In [40]:
print(kami_list)

[{'name': '［競走路の女王］オシリス', 'link': 'https://wikiwiki.jp/kamiprodb/%EF%BC%BB%E7%AB%B6%E8%B5%B0%E8%B7%AF%E3%81%AE%E5%A5%B3%E7%8E%8B%EF%BC%BD%E3%82%AA%E3%82%B7%E3%83%AA%E3%82%B9'}, {'name': '［HELIX］ニケ', 'link': 'https://wikiwiki.jp/kamiprodb/%EF%BC%BBHELIX%EF%BC%BD%E3%83%8B%E3%82%B1'}, {'name': '［春待ちの温もり］サマエル', 'link': 'https://wikiwiki.jp/kamiprodb/%EF%BC%BB%E6%98%A5%E5%BE%85%E3%81%A1%E3%81%AE%E6%B8%A9%E3%82%82%E3%82%8A%EF%BC%BD%E3%82%B5%E3%83%9E%E3%82%A8%E3%83%AB'}, {'name': '［正月福娘］フィア', 'link': 'https://wikiwiki.jp/kamiprodb/%EF%BC%BB%E6%AD%A3%E6%9C%88%E7%A6%8F%E5%A8%98%EF%BC%BD%E3%83%95%E3%82%A3%E3%82%A2'}, {'name': 'ヒュプノス［心想昇華］', 'link': 'https://wikiwiki.jp/kamiprodb/%E3%83%92%E3%83%A5%E3%83%97%E3%83%8E%E3%82%B9%EF%BC%BB%E5%BF%83%E6%83%B3%E6%98%87%E8%8F%AF%EF%BC%BD'}, {'name': '［省察の先に］茨木童子', 'link': 'https://wikiwiki.jp/kamiprodb/%EF%BC%BB%E7%9C%81%E5%AF%9F%E3%81%AE%E5%85%88%E3%81%AB%EF%BC%BD%E8%8C%A8%E6%9C%A8%E7%AB%A5%E5%AD%90'}, {'name': '［朧月の玉兎］イリス', 'link': 'https://wikiwiki.jp/k

In [41]:
import pandas as pd
df = pd.DataFrame(kami_list)
print(df)

              name                                               link
0     ［競走路の女王］オシリス  https://wikiwiki.jp/kamiprodb/%EF%BC%BB%E7%AB%...
1        ［HELIX］ニケ  https://wikiwiki.jp/kamiprodb/%EF%BC%BBHELIX%E...
2    ［春待ちの温もり］サマエル  https://wikiwiki.jp/kamiprodb/%EF%BC%BB%E6%98%...
3        ［正月福娘］フィア  https://wikiwiki.jp/kamiprodb/%EF%BC%BB%E6%AD%...
4      ヒュプノス［心想昇華］  https://wikiwiki.jp/kamiprodb/%E3%83%92%E3%83%...
..             ...                                                ...
101     不動明王［神化覚醒］  https://wikiwiki.jp/kamiprodb/%E4%B8%8D%E5%8B%...
102          アマテラス  https://wikiwiki.jp/kamiprodb/%E3%82%A2%E3%83%...
103    アマテラス［神化覚醒］  https://wikiwiki.jp/kamiprodb/%E3%82%A2%E3%83%...
104            アレス  https://wikiwiki.jp/kamiprodb/%E3%82%A2%E3%83%...
105      アレス［神化覚醒］  https://wikiwiki.jp/kamiprodb/%E3%82%A2%E3%83%...

[106 rows x 2 columns]


In [5]:
print(test_detail)

{'base': {'name': 'ソル', 'image': 'https://xn--hckqz0e9cygq471ahu9b.xn--wiki-4i9hs14f.com/index.php?plugin=ref&page=%E3%82%BD%E3%83%AB&src=%E3%82%BD%E3%83%AB2022.png', 'レアリティ': 'SSR', 'MaxHP/覚醒後': '1690', 'Max攻撃力': '6500', 'skills': [['ホワイト・プロミネンス+', '第3段階限界突破により解放される', '光属性ダメージ(特大)★性能UP'], ['太陽光炉+', '55', '6T', '-', '味方全体のHP回復(上限1500)/状態異常を1つ消去'], ['アールヴレズル', '-', '9T', '-', '敵単体に光属性ダメージ/敵の強化効果を1つ解除'], ['アールヴレズル+', '75', '8T', '-', '敵単体に光属性ダメージ/敵の強化効果を1つ解除★ターン短縮'], ['カルドルーチェ', '45', '7T', '180秒', '敵全体の攻撃DOWN(C枠/-20%)']], 'flavor': '恒星に匹敵するパワーを内包した女神。\nその明るい様子とは裏腹に苛烈な運命を背負っている。'}, 'awaken': {'name': 'ソル[神化覚醒]', 'image': 'https://xn--hckqz0e9cygq471ahu9b.xn--wiki-4i9hs14f.com/index.php?plugin=ref&page=%E3%82%BD%E3%83%AB&src=sol.png', 'MaxHP/覚醒後': '2500', 'Max攻撃力': '7500', 'skills': [['太陽光炉++', '65', '6T', '3T', '味方全体のHP回復(上限1800)/状態異常を2つ消去/リジェネ付与(250)'], ['アールヴレズル+', '-', '8T', '-', '敵単体に光属性ダメージ/敵の強化効果を1つ解除★ターン短縮'], ['アールヴレズル++', '75', '7T', '-', '敵全体に光属性ダメージ/敵の強化効果を1つ解除★ターン短縮'], ['カルドルー